In [0]:
# This cell is for setting up the environment and tools for data auditing in Databricks.
# 1. Install PySpark (not needed in Databricks, but shown for completeness)
# !pip install pyspark -q

# 2. Import tools (not needed in Databricks, as SparkSession and functions are available)
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, isnan, min, max, avg, sum

# 3. Initialize the Session (not needed in Databricks, as 'spark' is already available)
# spark = SparkSession.builder.master("local[*]").appName("DataAudit").getOrCreate()

Loading All Files

In [0]:
# Upload CSV files to the volume path in the bronze schema of captone_catalog catalog
import shutil

volume_path = "/Volumes/capstone_catalog/bronze/bronze_volume/"

shutil.copy("/Workspace/capstone_folder/sales_transactions.csv", volume_path + "sales_transactions.csv")
shutil.copy("/Workspace/capstone_folder/product_master.csv", volume_path + "product_master.csv")
shutil.copy("/Workspace/capstone_folder/store_master.csv", volume_path + "store_master.csv")
shutil.copy("/Workspace/capstone_folder/customer_data.csv", volume_path + "customer_data.csv")
shutil.copy("/Workspace/capstone_folder/inventory_data.csv", volume_path + "inventory_data.csv")
shutil.copy("/Workspace/capstone_folder/clickstream_events.csv", volume_path + "clickstream_events.csv")

sales_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(volume_path + "sales_transactions.csv")
prod_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(volume_path + "product_master.csv")
store_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(volume_path + "store_master.csv")
cust_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(volume_path + "customer_data.csv")
inv_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(volume_path + "inventory_data.csv")
click_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(volume_path + "clickstream_events.csv")

dfs = {
    "Sales": sales_df,
    "Products": prod_df,
    "Stores": store_df,
    "Customers": cust_df,
    "Inventory": inv_df,
    "Clickstream": click_df
}

In [0]:
# Create a list of your dataframes
dfs = {
    "Sales": sales_df,
    "Products": prod_df,
    "Stores": store_df,
    "Customers": cust_df,
    "Inventory": inv_df,
    "Clickstream": click_df
}

for name, df in dfs.items():
    print(f"--- Data Quality Audit: {name} ---")

    # 1. Count Total Rows
    print(f"Total Rows: {df.count()}")

    # 2. Check for Nulls in every column
    from pyspark.sql.functions import count, when, isnan, col
    from pyspark.sql.types import DoubleType, FloatType, IntegerType, LongType, ShortType, ByteType

    missing_value_expressions = []
    for c in df.columns:
        # Check if the column is of a numeric type that can have NaN (DoubleType or FloatType)
        if isinstance(df.schema[c].dataType, (DoubleType, FloatType)):
            missing_value_expressions.append(count(when(isnan(col(c)) | col(c).isNull(), c)).alias(c))
        else:
            missing_value_expressions.append(count(when(col(c).isNull(), c)).alias(c))
    df.select(missing_value_expressions).show()

    # 3. Check for Duplicates
    duplicate_count = df.count() - df.dropDuplicates().count()
    print(f"Duplicate Rows: {duplicate_count}")
    print("\n")

This code goes through every file and checks for:

Row Count: How much data do we have?

Null Values: Are there "holes" in our data?

Duplicates: Are we counting the same thing twice?

In [0]:
from pyspark.sql.functions import col, to_date, lit
from datetime import datetime
from pyspark.sql.types import DateType, TimestampType

print("--- Checking for Dates Greater Than Present Date ---")

# Get the current date and time
present_datetime = datetime.now()
present_date = present_datetime.date()

print(f"Comparing dates against: {present_date}")

dfs_to_check = {
    "Sales": (sales_df, ["order_date", "ingestion_timestamp"]),
    "Inventory": (inv_df, ["last_updated"]),
    "Clickstream": (click_df, ["event_timestamp"])
}

found_future_dates_overall = False

for df_name, (df, date_cols) in dfs_to_check.items():
    print(f"\n--- Checking DataFrame: {df_name} ---")
    found_future_dates_in_df = False

    for col_name in date_cols:
        # Ensure the column is a date or timestamp type before comparison
        # Convert to_date() if it's a timestamp for date-only comparison, or directly use if already date
        if col_name == "event_timestamp" and df_name == "Clickstream":
            # For clickstream, event_timestamp might be string, convert to timestamp first
            date_column = to_date(col(col_name))
        else:
            # For other DataFrames, assume schema inferencing has made them DateType
            date_column = col(col_name)

        # Filter for dates strictly greater than the present date
        future_records = df.filter(date_column > lit(present_date))

        if future_records.count() > 0:
            found_future_dates_overall = True
            found_future_dates_in_df = True
            print(f"⚠️ Found records in column '{col_name}' with dates greater than {present_date}:")
            future_records.select(col_name).distinct().show(truncate=False)

    if not found_future_dates_in_df:
        print(f"✅ No records with dates greater than {present_date} found in {df_name}.")

if not found_future_dates_overall:
    print("\nOverall: ✅ No records with dates greater than present date found across all checked DataFrames.")
else:
    print("\nOverall: ⚠️ Some records with dates greater than present date were found.")


### Outlier Detection in `quantity` using IQR (Original Raw Data)

This code identifies outliers in the `quantity` column of the *original* `sales_df` DataFrame using the Interquartile Range (IQR) method. Outliers are defined as values that fall below `Q1 - 1.5 * IQR` or above `Q3 + 1.5 * IQR`.

In [0]:
from pyspark.sql.functions import col

print("--- Checking for Outliers in 'quantity' column of ORIGINAL sales_df ---")

# Calculate Q1 and Q3 for 'quantity' in the original sales_df
quantiles_original = sales_df.approxQuantile("quantity", [0.25, 0.75], 0.05)
Q1_original = quantiles_original[0]
Q3_original = quantiles_original[1]
IQR_original = Q3_original - Q1_original

# Define outlier bounds
lower_bound_original = Q1_original - 1.5 * IQR_original
upper_bound_original = Q3_original + 1.5 * IQR_original

print(f"Q1 (original data): {Q1_original}")
print(f"Q3 (original data): {Q3_original}")
print(f"IQR (original data): {IQR_original}")
print(f"Lower Bound (original data): {lower_bound_original}")
print(f"Upper Bound (original data): {upper_bound_original}")

# Identify outliers
outliers_original_df = sales_df.filter(
    (col("quantity") < lower_bound_original) | (col("quantity") > upper_bound_original)
)

outliers_original_count = outliers_original_df.count()
print(f"\nTotal outliers found in 'quantity' column of ORIGINAL sales_df: {outliers_original_count}")

if outliers_original_count > 0:
    print("\nSample of records with outlier 'quantity' values in ORIGINAL sales_df:")
    outliers_original_df.select("transaction_id", "product_id", "quantity", "unit_price", "total_amount").show(20, truncate=False)
else:
    print("✅ No outliers found in the 'quantity' column of ORIGINAL sales_df based on the IQR method.")

checking for discount > 100%

### Referential Integrity Check: Orphan IDs in Sales Data (Original Raw Data)

This check identifies if any `product_id`, `store_id`, or `customer_id` in the *original* `sales_df` DataFrame are missing from their corresponding master dataframes (`prod_df`, `store_df`, `cust_df`).

A missing ID indicates an inconsistency, as a transaction would refer to a non-existent product, store, or customer.

In [0]:
print("--- Checking for Orphan IDs in ORIGINAL sales_df ---")

# Check for Sales with missing Product IDs in prod_df
orphan_products_count = sales_df.join(
    prod_df.select("product_id"), "product_id", "left_anti"
).count()
print(f"Sales records with unknown product_ids (not in prod_df): {orphan_products_count}")

# Check for Sales with missing Store IDs in store_df
orphan_stores_count = sales_df.join(
    store_df.select("store_id"), "store_id", "left_anti"
).count()
print(f"Sales records with unknown store_ids (not in store_df): {orphan_stores_count}")

# Check for Sales with missing Customer IDs in cust_df
orphan_customers_count = sales_df.join(
    cust_df.select("customer_id"), "customer_id", "left_anti"
).count()
print(f"Sales records with unknown customer_ids (not in cust_df): {orphan_customers_count}")

if orphan_products_count == 0 and orphan_stores_count == 0 and orphan_customers_count == 0:
    print("✅ All product_ids, store_ids, and customer_ids in ORIGINAL sales_df have matching entries in their respective master files.")
else:
    print("⚠️ Inconsistencies found in product_id, store_id, or customer_id referential integrity in ORIGINAL sales_df.")

### General City/State Mapping Logic Check in `store_df`

This code identifies inconsistencies in the `store_df` where a single city is mapped to more than one unique state, indicating potential data quality issues.

In [0]:
from pyspark.sql.functions import countDistinct, col

print("--- Checking General City/State Mapping in store_df ---")

# Group by city and count distinct states associated with each city
conflicting_city_state_mappings = store_df.groupBy("city") \
                                           .agg(countDistinct("state").alias("distinct_states")) \
                                           .filter("distinct_states > 1")

inconsistency_count = conflicting_city_state_mappings.count()

if inconsistency_count > 0:
    print(f"⚠️ Found {inconsistency_count} cities mapped to more than one state:")
    conflicting_city_state_mappings.show(truncate=False)
    print("\n--- Sample of records for these inconsistent cities ---")
    # Display actual records for these conflicting cities for further inspection
    store_df.join(conflicting_city_state_mappings, "city").orderBy("city").show(truncate=False)
else:
    print("✅ No cities found that are mapped to more than one state.")

### Cleaning: Standardizing City-State Mappings in `cleaned_store_master`

This code identifies the most frequent state for each city and then updates all records for that city in `cleaned_store_master` to use this canonical state, resolving inconsistencies where a single city was previously mapped to multiple states.

### Store Type Standardization Check in `store_df`

This code verifies that the `store_type` column in `store_df` contains only the allowed standardized values: 'Mall', 'Standalone', or 'Warehouse'. It will display any non-standard entries found.

In [0]:
from pyspark.sql.functions import col

print("\n--- Checking Store Type Standardization in store_df ---")

allowed_store_types = ["Mall", "Standalone", "Warehouse"]

non_standard_store_types = store_df.filter(~col("store_type").isin(allowed_store_types)).distinct()

non_standard_count = non_standard_store_types.count()

if non_standard_count > 0:
    print(f"⚠️ Found {non_standard_count} records with non-standard 'store_type' values:")
    non_standard_store_types.select("store_type").show(truncate=False)
else:
    print("✅ All 'store_type' values are standardized ('Mall', 'Standalone', or 'Warehouse').")

### Checking for Consistency in 'brand' Column (Raw Data)

This code identifies and displays records in the `prod_df` DataFrame where the `brand` column contains common inconsistent or placeholder values like 'None', 'N/A', or 'Unknown'.

In [0]:
from pyspark.sql.functions import col, upper

print("--- Checking 'brand' column for consistency issues in prod_df (Raw Data) ---")

# Define the list of inconsistent strings to check for
inconsistent_strings = ["None", "N/A", "Unknown", "none", "n/a", "unknown"]

# Filter the DataFrame for records where 'brand' matches any of the inconsistent strings
inconsistent_brands_df = prod_df.filter(upper(col("brand")).isin([s.upper() for s in inconsistent_strings]))

# Count and display the inconsistent records
inconsistent_brands_count = inconsistent_brands_df.count()

if inconsistent_brands_count > 0:
    print(f"⚠️ Found {inconsistent_brands_count} records with inconsistent 'brand' values:")
    inconsistent_brands_df.select("product_id", "brand").show(truncate=False)
else:
    print("✅ No inconsistent 'brand' values ('None', 'N/A', 'Unknown') found in prod_df.")

print("\n--- Displaying all distinct brand values for visual inspection ---")
prod_df.select("brand").distinct().show(truncate=False)

### Checking for Spelling Errors in 'category' Column (Raw Data)

This code extracts and displays all unique values from the `category` column in the `prod_df` DataFrame. This allows for a visual inspection to identify any potential spelling errors or inconsistencies in the raw category data.

In [0]:
print("--- Distinct 'category' values in prod_df (Raw Data) ---")
prod_df.select("category").distinct().show(truncate=False)

In [0]:
from pyspark.sql.functions import col

print("--- Checking for Discounts Greater Than 100% (i.e., discount > 1.0) in ORIGINAL sales_df ---")

discount_anomaly_df = sales_df.filter(col("discount") > 1.0)

anomaly_count = discount_anomaly_df.count()

if anomaly_count > 0:
    print(f"⚠️ Found {anomaly_count} records in ORIGINAL sales_df with discount values greater than 1.0:")
    discount_anomaly_df.select("transaction_id", "product_id", "quantity", "unit_price", "discount", "total_amount").show(20, truncate=False)
else:
    print("✅ No records found in ORIGINAL sales_df with discount values greater than 1.0.")

In [0]:
from pyspark.sql.types import DoubleType, FloatType

def run_basic_audit(df_dict):
    for name, df in df_dict.items():
        print(f"\n{'='*20} AUDITING: {name} {'='*20}")

        # Check 1: Total Volume
        total_rows = df.count()
        print(f"Total Records: {total_rows}")

        # Check 2: Missing Values (Nulls)
        # This scans every column and counts how many empty spots exist
        print("Missing Values per Column:")
        missing_value_expressions = []
        for c in df.columns:
            # Check if the column is of a numeric type (DoubleType or FloatType)
            if isinstance(df.schema[c].dataType, (DoubleType, FloatType)):
                missing_value_expressions.append(count(when(isnan(col(c)) | col(c).isNull(), c)).alias(c))
            else:
                missing_value_expressions.append(count(when(col(c).isNull(), c)).alias(c))
        df.select(missing_value_expressions).show()

        # Check 3: Duplicates
        unique_rows = df.dropDuplicates().count()
        if total_rows > unique_rows:
            print(f"⚠️ WARNING: Found {total_rows - unique_rows} duplicate rows!")
        else:
            print("✅ No duplicate rows found.")

run_basic_audit(dfs)

In [0]:
from pyspark.sql.types import IntegerType, DoubleType, FloatType, LongType, ShortType, ByteType

print("--- Checking for Negative Values in Numeric Columns ---")

for name, df in dfs.items():
    print(f"\n{'='*20} Checking DataFrame: {name} {'='*20}")
    found_negative = False
    for col_name in df.columns:
        # Check if the column is a numeric type
        if isinstance(df.schema[col_name].dataType, (IntegerType, DoubleType, FloatType, LongType, ShortType, ByteType)):
            negative_values_df = df.filter(col(col_name) < 0)
            if negative_values_df.count() > 0:
                found_negative = True
                print(f"\n⚠️ Negative values found in column '{col_name}':")
                negative_values_df.show(truncate=False)

    if not found_negative:
        print(f"✅ No negative values found in any numeric column of {name}.")

### Identifying Columns with Duplicate Values (Re-attempt)

This code will iterate through each DataFrame (`sales_df`, `prod_df`, etc.) and then through each column within those DataFrames. For every column, it calculates if the number of distinct values is less than the total number of rows, which indicates the presence of duplicate values within that specific column. This directly addresses your request to find columns that contain duplicate values.

In [0]:
from pyspark.sql.functions import countDistinct

print("\n--- Checking for Duplicate Values in Columns ---")

for name, df in dfs.items():
    print(f"\n{'='*20} Checking DataFrame: {name} {'='*20}")
    for col_name in df.columns:
        total_rows = df.count()
        distinct_values = df.select(countDistinct(col_name)).collect()[0][0]

        if total_rows > 0 and distinct_values < total_rows:
            print(f"⚠️ Column '{col_name}' has duplicate values ({distinct_values} distinct out of {total_rows} total rows).")
        else:
            print(f"✅ Column '{col_name}' has no duplicate values (all {total_rows} values are distinct or dataframe is empty).")


### Customers with conflicting age details

This code identifies `customer_id`s that have more than one unique age associated with them, indicating potential data inconsistencies where a single customer is recorded with different age details.

In [0]:
from pyspark.sql.functions import countDistinct, col

conflicting_age_customers = cust_df.groupBy("customer_id") \
                                   .agg(countDistinct("age").alias("distinct_ages")) \
                                   .filter("distinct_ages > 1")

print("Customers with conflicting age details (same ID, different ages):")
if conflicting_age_customers.count() > 0:
    conflicting_age_customers.show()
    print("Full details for customers with conflicting age information:")
    cust_df.join(conflicting_age_customers, "customer_id").orderBy("customer_id").show(truncate=False)
else:
    print("No customers found with conflicting age details.")

### Customers with conflicting details in any column

This code generalizes the check to find `customer_id`s that have more than one unique value for *any* descriptive column (excluding `customer_id` itself). This is a comprehensive way to detect inconsistencies in customer profiles.

In [0]:
from pyspark.sql.functions import countDistinct, col

print("\n--- Checking for Conflicting Details Across All Columns (per Customer ID) ---")

# Get all columns except 'customer_id'
other_columns = [c for c in cust_df.columns if c != 'customer_id']

found_conflicts = False
for column_name in other_columns:
    conflicting_data = cust_df.groupBy("customer_id") \
                                .agg(countDistinct(col(column_name)).alias(f"distinct_{column_name}")) \
                                .filter(f"distinct_{column_name} > 1")

    if conflicting_data.count() > 0:
        found_conflicts = True
        print(f"\n⚠️ Conflicts found in column '{column_name}' for customer IDs:")
        conflicting_data.show()
        # Optionally, show full details for these customers:
        # cust_df.join(conflicting_data, "customer_id").orderBy("customer_id").show(truncate=False)

if not found_conflicts:
    print("✅ No conflicting details found for any customer ID across all checked columns.")

To display dupliacte rows in each file

In [0]:
from pyspark.sql.functions import col, count, isnan, when

def run_basic_audit(df_dict):
    for name, df in df_dict.items():
        print(f"\n{'='*20} AUDITING: {name} {'='*20}")

        # ... (Your existing Total Volume and Missing Values checks) ...

        # Check 3: Displaying Duplicate Records
        # Group by all columns and count occurrences of each row
        duplicate_records = df.groupBy(df.columns) \
                              .count() \
                              .filter(col("count") > 1)

        if duplicate_records.count() > 0:
            print(f"⚠️ WARNING: Duplicate records found in {name}:")
            # Show the actual rows that are duplicated
            duplicate_records.show(truncate=False)
        else:
            print(f"✅ No duplicate rows found in {name}.")

run_basic_audit(dfs)

In [0]:
run_basic_audit(dfs)

A. Checking Sales Logic
We want to ensure no one bought "-5" items or got a price of "$0".

keep this for just reference

In [0]:
print("--- Sales Deep Dive ---")
sales_df.select(
    min("quantity").alias("Min_Qty"),
    max("quantity").alias("Max_Qty"),
    min("unit_price").alias("Min_Price"),
    sum(when(col("total_amount") <= 0, 1).otherwise(0)).alias("Invalid_Total_Amounts")
).show()

B. Checking Customer Ages
We want to see if any customer is 0 years old or 150 years old

In [0]:
print("--- Customer Age Check ---")
cust_df.select(
    min("age").alias("Youngest"),
    max("age").alias("Oldest"),
    avg("age").alias("Average_Age")
).show()

C. Checking Product Pricing
Does it ever happen that we sell a product for less than it cost us to buy it? (Cost > Selling Price).

In [0]:
print("--- Product Margin Audit ---")
# Count how many products are sold at a loss (Cost Price > Selling Price)
loss_making_products = prod_df.filter(col("cost_price") > col("selling_price")).count()
print(f"Number of products where Cost Price > Selling Price: {loss_making_products}")

print("\nSample of products where Cost Price > Selling Price:")
prod_df.filter(col("cost_price") > col("selling_price")) \
       .select("product_id", "cost_price", "selling_price").show()

D. Checking Inventory Health
Check for negative stock levels

In [0]:
print("--- Inventory Logic Check ---")
inv_df.filter(col("stock_on_hand") < 0).show()

In [0]:
#here invalid total is 0 but actual case it has be 50 also as price and quantity columns also have 50 such records
#it is solved while cleaning
print("--- Sales Logic Deep Dive ---")
# 1. Check for negative or zero values
sales_df.select(
    count(when(col("quantity") <= 0, 1)).alias("Invalid_Qty"),
    count(when(col("unit_price") <= 0, 1)).alias("Invalid_Price"),
    count(when(col("total_amount") <= 0, 1)).alias("Invalid_Total")
).show()

# this was calculates without considering discount factor

In [0]:
from pyspark.sql.functions import abs, col # Import abs from pyspark.sql.functions

# 2. Out-of-the-box: Check for math inconsistency
# (Allowing for a small rounding difference)
sales_df.withColumn("calculated_amt", col("quantity") * col("unit_price")) \
        .filter(abs(col("calculated_amt") - col("total_amount")) > 1) \
        .select("transaction_id", "total_amount", "calculated_amt") \
        .show(5)

### Corrected Math Inconsistency Check (including Discount)

Let's re-run the consistency check, this time correctly incorporating the `discount` into our calculated total.

here in abs if we put < 1 and check we might get a higher value that depicts values with very small difference that is less than 1 (0.4,0.5....) so we leave it consider for > 1 only


Exactly! In data analysis, it's very common to use a small tolerance (like your > 1 or sometimes > 0.01) when comparing floating-point numbers. This is because computers store and calculate numbers with decimals (like prices and discounts) using floating-point arithmetic, which can introduce tiny, unavoidable rounding errors.

In [0]:
from pyspark.sql.functions import abs, col

# Calculate the number of inconsistent transactions
inconsistent_count = sales_df.withColumn("calculated_total_with_discount",
                                      col("quantity") * col("unit_price") * (1 - col("discount"))) \
                               .filter(abs(col("calculated_total_with_discount") - col("total_amount")) > 2 ) \
                               .count()

print(f"Total number of inconsistent sales transactions: {inconsistent_count}")

while considering the total price I took it along with discount because , 1539 records matched calculated records with total records
but only 358 records are matching with the total_amount without taking the discount

In [0]:
from pyspark.sql.functions import abs, col

# Calculate the number consistent transactions without discount
consistent_count = sales_df.withColumn("calculated_total_without_discount",
                                      col("quantity") * col("unit_price") ) \
                               .filter(abs(col("calculated_total_without_discount") - col("total_amount")) ==0) \
                               .count()

print(f"Total number of consistent sales transactions: {consistent_count}")

///////////////////////////

### Inconsistent Sales Data (where `total_amount` != `quantity * unit_price * (1 - discount)`)

This check identifies transactions where the reported `total_amount` is mathematically inconsistent with the `quantity`, `unit_price`, and `discount` columns, beyond a small rounding tolerance.

this shows all data where our calculated amount not matches the total_amount given


In [0]:
from pyspark.sql.functions import abs, col

inconsistent_sales_data = sales_df.withColumn("calculated_total_with_discount",
                                      col("quantity") * col("unit_price") * (1 - col("discount"))) \
                               .filter(abs(col("calculated_total_with_discount") - col("total_amount")) > 1) \
                               .select("transaction_id", "quantity", "unit_price", "discount", "total_amount", "calculated_total_with_discount")

print(f"Number of inconsistent transactions: {inconsistent_sales_data.count()}")
inconsistent_sales_data.show(20, truncate=False)

This code returns the count of products which was selled at the right amount after applying the discount only 1539 are like that

In [0]:
from pyspark.sql.functions import abs, col

# 2. Out-of-the-box: Check for math inconsistency, now including discount
# (Allowing for a small rounding difference)
exact_match_count = sales_df.withColumn("calculated_total_with_discount",
                    col("quantity") * col("unit_price") * (1 - col("discount"))) \
        .filter(abs(col("calculated_total_with_discount") - col("total_amount")) == 0) \
        .count()

print(f"Number of transactions where calculated total exactly matches actual total: {exact_match_count}")

In [0]:
print("--- Customer Reality Check ---")
# 1. Check for impossible ages
cust_df.filter((col("age") < 18) | (col("age") > 100)).select("customer_id", "age").show()

# 2. Out-of-the-box: Consistency check
# Does a customer appear in Sales but not in Customer Master?
sales_custs = sales_df.select("customer_id").distinct()
missing_custs = sales_custs.join(cust_df, "customer_id", "left_anti").count()
print(f"Transactions from unknown customers: {missing_custs}")

In [0]:
print("--- Margin & Stock Audit ---")
# 1. Negative Margin Check
prod_df.filter(col("selling_price") < col("cost_price")) \
       .select("product_id", "cost_price", "selling_price").show()

# 2. Impossible Stock
inv_df.filter(col("stock_on_hand") < 0).show()

In [0]:
print("--- Platform Standardization Check ---")
click_df.groupBy("platform").count().show()
# Look for: 'Web', 'web', 'WEB' - these should be the same!
#on analyzing it is common only

### Customers with conflicting city details

This code identifies `customer_id`s that have more than one unique city associated with them, indicating potential data inconsistencies where a single customer is recorded with different city details.
so on checking no such fields

In [0]:
from pyspark.sql.functions import countDistinct

conflicting_city_customers = cust_df.groupBy("customer_id") \
                                    .agg(countDistinct("city").alias("distinct_cities")) \
                                    .filter("distinct_cities > 1")

# Show the customer IDs with conflicting city information
print("Customers with conflicting city details (same ID, different cities):")
conflicting_city_customers.show()

# To see the full details of these customers:
print("Full details for customers with conflicting city information:")
cust_df.join(conflicting_city_customers, "customer_id").orderBy("customer_id").show(truncate=False)

A. ID Integrity (The "Ghost" Check)
Check if there are Sales for products or stores that do not exist in your Master files. If a product_id is in Sales but not in Product Master, you won't know the category or cost.

In [0]:
# Check for Sales with missing Product IDs
ghost_products = sales_df.join(prod_df, "product_id", "left_anti").count()
print(f"Sales records with unknown Product IDs: {ghost_products}")

# Check for Sales with missing Store IDs
ghost_stores = sales_df.join(store_df, "store_id", "left_anti").count()
print(f"Sales records with unknown Store IDs: {ghost_stores}")

B. Date Logic (The "Time Traveler" Check)
Ensure that the ingestion_timestamp (when data entered the system) is not before the order_date.

In [0]:
# Count records where ingestion happened before the actual sale
time_travelers = sales_df.filter(col("ingestion_timestamp") < col("order_date")).count()
print(f"Records with impossible date logic: {time_travelers}")

C. Loyalty "None" Check
In your customer_data.csv, the status often shows the word "None". Spark might not see this as a Null because it's a string. Let's see how many "None" strings exist.

In [0]:
cust_df.groupBy("loyalty_status").count().show()

In [0]:
# Store sales_df as a Delta table in capstone_catalog.bronze.sales
sales_df.write.format("delta").mode("overwrite").saveAsTable("capstone_catalog.bronze.sales")

# Store cust_df as a Delta table in capstone_catalog.bronze.customers
cust_df.write.format("delta").mode("overwrite").saveAsTable("capstone_catalog.bronze.customers")

# Store prod_df as a Delta table in capstone_catalog.bronze.products
prod_df.write.format("delta").mode("overwrite").saveAsTable("capstone_catalog.bronze.products")

# Store inv_df as a Delta table in capstone_catalog.bronze.inventory
inv_df.write.format("delta").mode("overwrite").saveAsTable("capstone_catalog.bronze.inventory")

# Store store_df as a Delta table in capstone_catalog.bronze.stores
store_df.write.format("delta").mode("overwrite").saveAsTable("capstone_catalog.bronze.stores")

# Store click_df as a Delta table in capstone_catalog.bronze.clicks
click_df.write.format("delta").mode("overwrite").saveAsTable("capstone_catalog.bronze.clicks")

In [0]:
# Option 1: Reload all DataFrames from Delta tables
# This replaces the CSV-based DataFrames with Delta table-based ones

sales_df = spark.table("capstone_catalog.bronze.sales")
cust_df = spark.table("capstone_catalog.bronze.customers")
prod_df = spark.table("capstone_catalog.bronze.products")
inv_df = spark.table("capstone_catalog.bronze.inventory")
store_df = spark.table("capstone_catalog.bronze.stores")
click_df = spark.table("capstone_catalog.bronze.clicks")

print("✅ All DataFrames now loaded from Delta tables")
print(f"Sales records: {sales_df.count()}")

//////////////////////////////